In [2]:
!pip install requests
!pip install mysql-connector-python

In [ ]:
import requests
import json
import time
import re
import mysql.connector

In [4]:
# Connecting tio database

conn_mysql=mysql.connector.connect(
    host="localhost",
    user="root",
    password="root"
)
cursor_mysql=conn_mysql.cursor()
print("Connection Established")

Connection Established


In [5]:
cursor_mysql.execute("use cricbuzz_livestats_project")

In [39]:
cursor_mysql.execute("""
CREATE TABLE matches (
    matchId BIGINT PRIMARY KEY,
    matchType VARCHAR(50),
    seriesId BIGINT,
    matchDesc VARCHAR(255),
    matchStatus VARCHAR(255),
    matchFormat VARCHAR(50),
    startDate BIGINT,
    endDate BIGINT,
    state VARCHAR(50),
    team1Id BIGINT,
    team2Id BIGINT,
    venueId BIGINT,
    winningTeamName VARCHAR(100),
    victoryMargin INT,
    victoryType VARCHAR(20),
    tossStatus VARCHAR(255) DEFAULT 'Not Available'
);
""")

conn_mysql.commit()
print("Matches table created successfully")

Matches table created successfully


In [ ]:
# Batsman Table
cursor_mysql.execute("""
CREATE TABLE IF NOT EXISTS batsman_scores (
    matchId BIGINT NOT NULL,
    inningsId BIGINT NOT NULL,
    batsmanId BIGINT NOT NULL,
    batsmanName VARCHAR(100),
    balls INT,
    runs INT,
    fours INT,
    sixes INT,
    strikeRate FLOAT,
    isCaptain BOOLEAN,
    isKeeper BOOLEAN,
    outDescription VARCHAR(255),

    PRIMARY KEY (matchId, inningsId, batsmanId),
    FOREIGN KEY (matchId) REFERENCES matches(matchId)
);
""")

# Bowler Table
cursor_mysql.execute("""
CREATE TABLE IF NOT EXISTS bowler_scores (
    matchId BIGINT NOT NULL,
    inningsId BIGINT NOT NULL,
    bowlerId BIGINT NOT NULL,
    bowlerName VARCHAR(100),
    overs VARCHAR(10),
    maidens INT,
    wickets INT,
    runs INT,
    economy FLOAT,
    dots INT,
    balls INT,
    isCaptain BOOLEAN,
    isKeeper BOOLEAN,

    PRIMARY KEY (matchId, inningsId, bowlerId),

    FOREIGN KEY (matchId) REFERENCES matches(matchId)
);
""")

# Partnership Table
cursor_mysql.execute("""
CREATE TABLE IF NOT EXISTS partnerships (
    matchId BIGINT NOT NULL,
    inningsId BIGINT NOT NULL,
    partnershipId BIGINT NOT NULL,

    bat1Id BIGINT,
    bat1Name VARCHAR(100),
    bat1Runs INT,
    bat1Fours INT,
    bat1Sixes INT,
    bat1Balls INT,

    bat2Id BIGINT,
    bat2Name VARCHAR(100),
    bat2Runs INT,
    bat2Fours INT,
    bat2Sixes INT,
    bat2Balls INT,

    totalRuns INT,
    totalBalls INT,
    teamName VARCHAR(100),
    teamId BIGINT,

    PRIMARY KEY (matchId, inningsId, partnershipId),

    FOREIGN KEY (matchId) REFERENCES matches(matchId)
);
""")



In [41]:
cursor_mysql.execute("""
        CREATE TABLE venues (
        venueId INT PRIMARY KEY,
        ground VARCHAR(150),
        city VARCHAR(100),
        country VARCHAR(100),
        timezone VARCHAR(20),
        capacity INT,
        ends VARCHAR(150),
        home_team VARCHAR(100))
        """)

In [42]:
cursor_mysql.execute("""
CREATE TABLE player_career_bowling (
player_id BIGINT,
format VARCHAR(10),
matches INT,
wickets INT,
average FLOAT,
economy FLOAT
        );
        """)

In [43]:
cursor_mysql.execute("""
        CREATE TABLE player_career_batting (
        player_id BIGINT,
        format VARCHAR(10),
        matches INT,
        runs INT,
        average FLOAT,
        hundreds INT
        );
        """)

In [44]:
cursor_mysql.execute("""
    CREATE TABLE IF NOT EXISTS teams (
    team_id INT PRIMARY KEY,
    team_name VARCHAR(100) NOT NULL,
    short_name VARCHAR(20),
    country_name VARCHAR(100),
    category VARCHAR(100));""")
conn_mysql.commit()


In [45]:
cursor_mysql.execute("""CREATE TABLE players (
    player_id BIGINT PRIMARY KEY,
    name VARCHAR(150),
    nickname VARCHAR(150),
    role VARCHAR(100),
    batting_style VARCHAR(100),
    bowling_style VARCHAR(100),
    birth_place VARCHAR(150),
    date_of_birth VARCHAR(100),
    international_team VARCHAR(100));""")

In [ ]:
# #droping all tables
# cursor_mysql.execute("""DROP TABLE IF EXISTS partnerships;
# DROP TABLE IF EXISTS batsman_scores;
# DROP TABLE IF EXISTS bowler_scores;

# DROP TABLE IF EXISTS player_career_batting;
# DROP TABLE IF EXISTS player_career_bowling;

# DROP TABLE IF EXISTS players;

# DROP TABLE IF EXISTS matches;

# DROP TABLE IF EXISTS venues;
# DROP TABLE IF EXISTS teams;
# DROP TABLE IF EXISTS series;""")

In [ ]:
cursor_mysql.execute("""CREATE TABLE series (
    series_id BIGINT PRIMARY KEY,
    series_name VARCHAR(255),
    start_date BIGINT,
    end_date BIGINT
);""")

In [ ]:
def fillvenue(venueId):

    url = "https://cricbuzz-cricket.p.rapidapi.com/venues/v1/"+ str(venueId)

    headers = {
        "x-rapidapi-key": "211a4cb2c7mshef258cdb418b086p116d87jsnd08701bab594",
        "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers)

    venue=response.json()
        
    venueground=venue.get("ground","")
    venuecity=venue.get("city","")
    venuecountry = venue.get("country","")
    venuetimezone = venue.get("timezone")
    venuecapacity = int(re.sub(r"\D","",venue.get("capacity", '0')))
    # venueends = venue["ends"]
    venuehomeTeam = venue.get("homeTeam", "")  
    
    cursor_mysql.execute("""
    INSERT IGNORE INTO venues
    (venueId, ground, city, country, timezone, capacity, home_team)
    VALUES (%s, %s, %s, %s, %s, %s, %s)""", 
    (venueId, venueground, venuecity, venuecountry, venuetimezone,
    venuecapacity, venuehomeTeam))
    conn_mysql.commit()

In [ ]:
def fillseries(seriesId1):
    
    url = "https://cricbuzz-cricket.p.rapidapi.com/series/v1/" +str(seriesId1)


    headers = {
        "x-rapidapi-key": "211a4cb2c7mshef258cdb418b086p116d87jsnd08701bab594",
        "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers)
    data = response.json()

    # extracting series name from first match
    series_name = None
    start_date = None
    end_date = None

    for item in data.get("matchDetails", []):
        if "matchDetailsMap" in item:
            match_list = item["matchDetailsMap"].get("match", [])
            if match_list:
                matchInfo = match_list[0]["matchInfo"]
                series_name = matchInfo.get("seriesName")
                start_date = matchInfo.get("startDate")
                end_date = matchInfo.get("endDate")
                break

    cursor_mysql.execute("""
        INSERT IGNORE INTO series
        (series_id, series_name, start_date, end_date)
        VALUES (%s,%s,%s,%s)
    """, (
        seriesId1,
        series_name,
        start_date,
        end_date
    ))

    conn_mysql.commit()
    print("Inserted Series:", seriesId1)

In [ ]:
def update_toss_status(match_id):
    url = "https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/" + str(match_id)

    headers = {
        "x-rapidapi-key": "211a4cb2c7mshef258cdb418b086p116d87jsnd08701bab594",   
        "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers)
    data = response.json()

    # extracting toss status
    toss_status = data.get("tossstatus", "")

    if not toss_status:
        toss_status = "Not Available"

    # updating only toss columns
    cursor_mysql.execute("""
        UPDATE matches
        SET tossStatus=%s
        WHERE matchId=%s
    """, (
        toss_status,
        match_id
    ))
    conn_mysql.commit()

    print("Toss Updated:", match_id, toss_status)

In [ ]:
def get_player(player_id):

    url = "https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/" + str(player_id)

    headers = {
        "x-rapidapi-key": "211a4cb2c7mshef258cdb418b086p116d87jsnd08701bab594",
        "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers)

    return response.json()
    print(json.dump(response.json()))
    
    

In [ ]:
def get_player_batting(player_id):

    url = f"https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/{player_id}/batting"

    headers = {
        "x-rapidapi-key": "211a4cb2c7mshef258cdb418b086p116d87jsnd08701bab594",
        "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers)

    return response.json()

In [ ]:
def get_player_bowling(player_id):

    url = f"https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/{player_id}/bowling"

    headers = {
        "x-rapidapi-key": "211a4cb2c7mshef258cdb418b086p116d87jsnd08701bab594",
        "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers)
    return response.json()

In [ ]:
def fillData(recent_data):
    
    venues = set()
    matchesid=[]
    series_ids = set()

    for typeMatch in recent_data["typeMatches"][0:6]:
        matchType=typeMatch["matchType"]

        for seriesMatch in typeMatch["seriesMatches"]:
            if "seriesAdWrapper" in seriesMatch:
                seriesAdWrapper= seriesMatch["seriesAdWrapper"]

                for match in seriesAdWrapper["matches"]:

                    matchId=match["matchInfo"]["matchId"]
                    seriesId=match["matchInfo"]["seriesId"]
                    matchDesc=match["matchInfo"]["matchDesc"]
                    status=match["matchInfo"]["status"]
                    matchFormat=match["matchInfo"]["matchFormat"]
                    startDate=match["matchInfo"]["startDate"]
                    endDate=match["matchInfo"]["endDate"]
                    state=match["matchInfo"]["state"]

                    team1Id=match["matchInfo"]["team1"]["teamId"]
                    team2Id=match["matchInfo"]["team2"]["teamId"]

                    venueInfo=match["matchInfo"]["venueInfo"]
                    venueId=venueInfo["id"]

                    winning_team = None
                    victory_margin = None
                    victory_type = None

                    print(status)

                    if status:
                        status_lower = status.lower()

                        # if match is DRAW
                        if "drawn" in status_lower:
                            winning_team = "Draw"
                            victory_margin = 0
                            victory_type = "draw"

                        # match is TIE
                        elif "tied" in status_lower:
                            winning_team = "Tie"
                            victory_margin = None
                            victory_type = None
                            
                        elif "rain" in status_lower:
                            winning_team = "Mathch Postponed"
                            victory_margin = 0
                            victory_type = "Mathch Postponed"


                        # if there have NO RESULT / ABANDON / CANCEL
                        elif "no result" in status_lower or "abandon" in status_lower or "cancel" in status_lower:
                            winning_team = "No Result"
                            victory_margin = 0
                            victory_type = "NO RESULT"

                        # IF THERE HAVE NORMAL WIN
                        elif "won by" in status_lower:
                            parts = status.split(" won by ")
                            if len(parts) == 2:
                                winning_team = parts[0].strip()
                                margin_part = parts[1]

                                # Extract number
                                margin_number = re.findall(r"\d+", margin_part)
                                if margin_number:
                                    victory_margin = int(margin_number[0])

                                # Detect type
                                if "run" in margin_part.lower():
                                    victory_type = "runs"
                                elif "wicket" in margin_part.lower() or "wkts" in margin_part.lower() or "wkt" in margin_part.lower():
                                    victory_type = "wickets"
                    # CONDITION FOR ONLY 5 VENUES
                    if len(venues) < 5:
                        venues.add(venueId)

                    if len(matchesid) < 5:
                        matchesid.append(matchId)
                        
                    if len(series_ids)<5:
                        series_ids.add(seriesId)

                    cursor_mysql.execute("""
                        INSERT IGNORE INTO matches
                        (matchId, matchType, seriesId, matchDesc, matchStatus, matchFormat,
                         startDate, endDate, state, team1Id, team2Id, venueId,
                         winningTeamName, victoryMargin, victoryType)
                        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                    """, (
                        matchId, matchType, seriesId, matchDesc, status, matchFormat,
                        startDate, endDate, state, team1Id, team2Id, venueId,
                        winning_team, victory_margin, victory_type
                    ))

    conn_mysql.commit()
    print(venues)

    for venueId in set(venues):
        fillvenue(venueId)
    for seriesId1 in set(series_ids):
        fillseries(seriesId1)

    return matchesid

In [ ]:
for option in ['recent']: #, 'live', 'upcoming']:
	url = "https://cricbuzz-cricket.p.rapidapi.com/matches/v1/" + option

	headers = {
		"x-rapidapi-key": "211a4cb2c7mshef258cdb418b086p116d87jsnd08701bab594",
		"x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
	}
	response = requests.get(url, headers=headers)

	recent_data=response.json()
	match_ids = fillData(recent_data)
	for match_id in match_ids:
		update_toss_status(match_id)
	print("Results updated")

Bangladesh won by 11 runs
Pakistan won by 128 runs  -  2nd inns reduced to 32 overs (DLS Target 243)
Bangladesh won by 8 wkts
South Africa won by 7 wkts
Match abandoned without toss
Match abandoned without toss
Cyprus won by 48 runs
Bahrain won by 7 wkts  -  DLS Method (Target : 132)
Cayman Islands won by 9 wkts
Argentina won by 92 runs
Argentina won by 7 wkts
Cayman Islands won by 92 runs
Argentina won by 62 runs
Cayman Islands won by 9 wkts
Cayman Islands won by 9 wkts
Argentina won by 15 runs
Mexico won by 9 runs
Cayman Islands won by 10 wkts
Botswana won by 104 runs
Botswana won by 6 runs
Botswana won by 6 wkts
Botswana won by 6 wkts
AWS World won by 50 runs
Mumbai Spartans won by 17 runs
Konark Suryas Odisha won by 18 runs
India Tigers won by 20 runs
India Captains won by 23 runs
North West  -  Eastvaal Renault Dragons won by 134 runs
Warriors won by 87 runs
Titans won by 6 wkts  -  20 overs game due to rain
Western Province won by 3 wkts
Lions won by 7 wkts
Warriors won by 11 run

In [ ]:
for match_id in match_ids:
    print(match_id)
    url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}/scard"

    headers = {
        "x-rapidapi-key": "211a4cb2c7mshef258cdb418b086p116d87jsnd08701bab594",
        "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers)

    score_card=response.json()

    for innings in score_card["scorecard"][0:6]:
        
        scoreinningsid = innings["inningsid"]
        
        for batsman in innings["batsman"]:
            
            batsmanid = batsman["id"]
            batsmanballs = batsman["balls"]
            batsmanruns = batsman["runs"]
            batsmanfours = batsman["fours"]
            batsmansixes = batsman["sixes"]
            batsmanstrike_rate = batsman["strkrate"]
            batsmanname = batsman["name"]
            batsmaniscaptain = batsman["iscaptain"]
            batsmaniskeeper = batsman["iskeeper"]
            batsmanoutdescription = batsman["outdec"]
            
            cursor_mysql.execute("""
            INSERT IGNORE INTO batsman_scores
            (matchId, inningsId, batsmanId, batsmanName, balls, runs,
            fours, sixes, strikeRate, isCaptain,
            isKeeper, outDescription)
            VALUES (%s, %s, %s, %s, %s,
                    %s, %s, %s, %s,
                    %s, %s, %s)""", (
                match_id,
                scoreinningsid,
                batsman["id"],
                batsman["name"],
                batsman["balls"],
                batsman["runs"],
                batsman["fours"],
                batsman["sixes"],
                batsman["strkrate"],
                batsman["iscaptain"],
                batsman["iskeeper"],
                batsman["outdec"]
            ))
            
        for bowler in innings["bowler"]:
                
            bowlerid = bowler["id"]
            bowlername = bowler["name"]
            bowlerovers = bowler["overs"]
            bowlermaidens = bowler["maidens"]
            bowlerwickets = bowler["wickets"]
            bowlerruns = bowler["runs"]
            bowlereconomy = bowler["economy"]
            bowlerdots = bowler["dots"]
            bowlerballs = bowler["balls"]
            bowleriscaptain = bowler["iscaptain"]
            bowleriskeeper = bowler["iskeeper"]
            
            cursor_mysql.execute("""
            INSERT IGNORE INTO bowler_scores
            (matchId,inningsId, bowlerId, bowlerName, overs, maidens,
            wickets, runs, economy, dots, balls,
            isCaptain, isKeeper)
            VALUES (%s, %s, %s, %s, %s,
                    %s, %s, %s, %s, %s,
                    %s, %s,%s )""", (
                match_id,
                scoreinningsid,
                bowler["id"],
                bowler["name"],
                bowler["overs"],
                bowler["maidens"],
                bowler["wickets"],
                bowler["runs"],
                bowler["economy"],
                bowler["dots"],
                bowler["balls"],
                bowler["iscaptain"],
                bowler["iskeeper"],
            ))
        
        for part in innings["partnership"]["partnership"]:

            partnership_id = part["id"]
            bat1id = part["bat1id"]
            bat1name = part["bat1name"]
            bat1runs = part["bat1runs"]
            bat1fours = part["bat1fours"]
            bat1sixes = part["bat1sixes"]

            bat2id = part["bat2id"]
            bat2name = part["bat2name"]
            bat2runs = part["bat2runs"]
            bat2fours = part["bat2fours"]
            bat2sixes = part["bat2sixes"]

            totalruns = part["totalruns"]
            totalballs = part["totalballs"]

            bat1balls = part["bat1balls"]
            bat2balls = part["bat2balls"]

            teamname = part["teamname"]
            teamid = part["teamid"]
            
            cursor_mysql.execute("""
            INSERT IGNORE INTO partnerships
            (matchId,inningsId, partnershipId,
            bat1Id, bat1Name, bat1Runs, bat1Fours, bat1Sixes, bat1Balls,
            bat2Id, bat2Name, bat2Runs, bat2Fours, bat2Sixes, bat2Balls,
            totalRuns, totalBalls, teamName, teamId)
            VALUES (%s, %s,%s,
                    %s, %s, %s, %s, %s, %s,
                    %s, %s, %s, %s, %s, %s,
                    %s, %s, %s, %s)""", (
                match_id,
                scoreinningsid,
                part["id"],
                part["bat1id"], part["bat1name"], part["bat1runs"],
                part["bat1fours"], part["bat1sixes"], part["bat1balls"],
                part["bat2id"], part["bat2name"], part["bat2runs"],
                part["bat2fours"], part["bat2sixes"], part["bat2balls"],
                part["totalruns"], part["totalballs"],
                part["teamname"], part["teamid"]
            ))

    conn_mysql.commit()

147885
147874
147863
122687
149600


In [ ]:
for option in ["international"]: #,"league","domestic","women"]:
    url = "https://cricbuzz-cricket.p.rapidapi.com/teams/v1/" + option
    headers = {
	"x-rapidapi-key": "211a4cb2c7mshef258cdb418b086p116d87jsnd08701bab594",
	"x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
    }
    response = requests.get(url, headers=headers)
    teams=response.json()
    print(json.dumps(teams))
    for team in teams["list"]:
        if "teamId" not in team:
            currentcategory = team["teamName"]
            continue
        teamid = team["teamId"]
        teamname = team["teamName"]
        shortname = team.get("teamSName", "")
        countryname = team.get("countryName", "")
        
        cursor_mysql.execute("""
        INSERT IGNORE INTO teams
        (team_id, team_name, short_name, country_name, category)
        VALUES (%s, %s, %s, %s, %s)""", 
        (teamid, teamname, shortname, countryname, currentcategory))
        
    conn_mysql.commit()

{"list": [{"teamName": "Test Teams"}, {"teamId": 2, "teamName": "India", "teamSName": "IND", "imageId": 776162, "countryName": "India"}, {"teamId": 96, "teamName": "Afghanistan", "teamSName": "AFG", "imageId": 776177}, {"teamId": 27, "teamName": "Ireland", "teamSName": "IRE", "imageId": 839366}, {"teamId": 3, "teamName": "Pakistan", "teamSName": "PAK", "imageId": 776308, "countryName": "Pakistan"}, {"teamId": 4, "teamName": "Australia", "teamSName": "AUS", "imageId": 776202, "countryName": "Australia"}, {"teamId": 5, "teamName": "Sri Lanka", "teamSName": "SL", "imageId": 776254, "countryName": "Sri Lanka"}, {"teamId": 6, "teamName": "Bangladesh", "teamSName": "BAN", "imageId": 776210}, {"teamId": 9, "teamName": "England", "teamSName": "ENG", "imageId": 776237}, {"teamId": 10, "teamName": "West Indies", "teamSName": "WI", "imageId": 776191}, {"teamId": 11, "teamName": "South Africa", "teamSName": "RSA", "imageId": 776287}, {"teamId": 12, "teamName": "Zimbabwe", "teamSName": "ZIM", "imag

In [18]:
processed_players = set() # this is for avoiding duplicates

for innings in score_card["scorecard"][:6]:

    # BATTING PLAYERS
    for batsman in innings["batsman"]:
        player_id = int(batsman["id"])

        if player_id not in processed_players:
            # Get batting career
            batting_data = get_player_batting(player_id)

            formats = batting_data["headers"][1:]  # Test, ODI, T20, IPL
            rows = batting_data["values"]

            # Creatinge a dictionary
            stats = {}

            for row in rows:
                row_name = row["values"][0]
                row_values = row["values"][1:]
                stats[row_name] = row_values

            # Inserting format-wise enumerate means it will have default counter i=0,1,2,3....
            for i, format_name in enumerate(formats):

                cursor_mysql.execute("""
                    INSERT IGNORE INTO player_career_batting
                    (player_id, format, matches, runs, average, hundreds)
                    VALUES (%s,%s,%s,%s,%s,%s)
                """, (
                    player_id,
                    format_name,
                    stats["Matches"][i],
                    stats["Runs"][i],
                    stats["Average"][i],
                    stats["100s"][i]
                ))

            conn_mysql.commit()

            player_data = get_player(player_id)

            cursor_mysql.execute("""
                INSERT IGNORE INTO players
                (player_id, name, nickname, role, batting_style,
                bowling_style, birth_place, date_of_birth, international_team)
                VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
            """, (
                player_data.get("id"),
                player_data.get("name"),
                player_data.get("nickName"),
                player_data.get("role"),
                player_data.get("bat"),
                player_data.get("bowl"),
                player_data.get("birthPlace"),
                player_data.get("DoBFormat"),
                player_data.get("intlTeam"),
            ))
            processed_players.add(player_id)


    # BOWLING PLAYERS
    for bowler in innings["bowler"]:
        player_id = int(bowler["id"])

        if player_id not in processed_players:
            
            bowling_data = get_player_bowling(player_id)

            formats = bowling_data["headers"][1:]   # Test, ODI, T20, IPL
            rows = bowling_data["values"]

            # Createing a dictionary
            stats = {}

            for row in rows:
                row_name = row["values"][0]
                row_values = row["values"][1:]
                stats[row_name] = row_values

            for i, format_name in enumerate(formats):

                cursor_mysql.execute("""
                    INSERT INTO player_career_bowling
                    (player_id, format, matches, wickets, average, economy)
                    VALUES (%s,%s,%s,%s,%s,%s)
                """, (
                    player_id,
                    format_name,
                    stats["Matches"][i],
                    stats["Wickets"][i],
                    stats["Avg"][i],
                    stats["Eco"][i]
                ))

            conn_mysql.commit()

            player_data = get_player(player_id)

            cursor_mysql.execute("""
            INSERT IGNORE INTO players
            (player_id, name, nickname, role, batting_style,
            bowling_style, birth_place, date_of_birth, international_team)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)""", 
            (
            player_data.get("id"),
            player_data.get("name"),
            player_data.get("nickName"),
            player_data.get("role"),
            player_data.get("bat"),
            player_data.get("bowl"),
            player_data.get("birthPlace"),
            player_data.get("DoBFormat"),
            player_data.get("intlTeam"),))

            processed_players.add(player_id)


conn_mysql.commit()